# HiBASIL Tutorial 2: Solving the Spatial Mixture
This notebook introduces the BASIL (BAyesian Source Inference and Localization) framework. We will simulate a disease outbreak from two unknown sources and use Bayesian inference to separate two verlapping signals and infer their mixture weights.

## 1. Environment Setup
First, we prepare the workspace by creating directories and loading the necessary Bayesian engines.

In [1]:
import os
import numpy as np
import pandas as pd
# Set environment flags for PyTensor
os.environ['PYTENSOR_FLAGS'] = "base_compiledir=./pytensor_cache,cxx="

import pymc as pm
import arviz as az
from tqdm.auto import tqdm

import matplotlib
matplotlib.use('Agg')  # or 'Qt5Agg'
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score

# Create necessary directories automatically
for folder in ['./sim', './output', './pytensor_cache']:
    os.makedirs(folder, exist_ok=True)

import simulation  # simulation.py developed in this study
import hibasil       # core functions of the HiBASIL framework

np.random.seed(42)

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
C:\Users\sunny\anaconda3\envs\mcenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ PyTensor cache: pytensor_cache


## 2. Generating the "Mystery" Data
We simulate a two-foci outbreak using a Power-Law dispersal kernel. Note the use of eps = 1e-12 to ensure numerical stability.

In [2]:
sim_dir = "./sim/"
output_dir = "./output/"

eps = 1e-08
p0 = 0.3
q_param = 0.01  # constant value for p1
phi = 18
model_type = 'power_law'

# focus 1
scale1 = 5.0
exponent1 = 2.0
fx1_obs = 0.0  # Focus 1 X for each observation
fy1_obs = 0.0  # Focus 1 Y for each observation
fz1_obs = 0.6  # Focus 1 intensity for each observation
focus1 = [fx1_obs, fy1_obs, fz1_obs]
weight = 0.6

# focus 2
scale2 = 5.0
exponent2 = 2.0
fx2_obs = 0.0  # Focus 1 X for each observation
fy2_obs = 50.0  # Focus 1 Y for each observation
fz2_obs = 0.8  # Focus 1 intensity for each observation
focus2 = [fx2_obs, fy2_obs, fz2_obs]

foci = [fx1_obs, fy1_obs, fz1_obs, fx2_obs, fy2_obs, fz2_obs]

# Define the varying coordinates
y_coords = np.linspace(start=-2, stop=52, num=250)

# ensure focus location is in the coords list
y_coords = np.unique(np.concatenate([[fy1_obs], y_coords, [fy2_obs]]))

x_coords = np.full(y_coords.shape, 0.0) # X is always 0.0
sample_points = np.column_stack((x_coords, y_coords))  # np.column_stack puts the arrays side-by-side

sample_kernel = simulation.two_foci_kernel(sample_points[:,0], sample_points[:, 1], focus1, focus2, scale1, 
                                scale2, weight, model_type=model_type, 
                                exponent1=exponent1, exponent2=exponent2)

# calculate zero-inflation probability
p_param = np.clip(p0 * (1- sample_kernel), eps, 1- eps)
mu = np.clip(sample_kernel, eps, 1- eps)

alpha_samples = mu * phi
beta_samples = (1 - mu) * phi

prob_zero = p_param
prob_one = (1.0 - p_param) * q_param
prob_continuous = (1.0 - p_param) * (1.0 - q_param)

# simulate a four-row disease dataset
n_rows = 4
obs_df = pd.DataFrame({
    'InterrowDistance': pd.Series(dtype='float64'),
    'Distance': pd.Series(dtype='float64'),
    'Severity': pd.Series(dtype='float64'),
    'Row': pd.Series(dtype='int64'),
    'Plant': pd.Series(dtype='object')
})
for run_id in range(n_rows):
    y_obs_run = simulation.sampling(sample_points, foci, prob_zero, prob_one, prob_continuous, alpha_samples, beta_samples)
    row_val = run_id%n_rows
    row_label = np.full(len(sample_points), row_val)
    y_obs_run['Row'] = row_label
    
    obs_df = pd.concat([obs_df, y_obs_run])        

print(obs_df.head())
obs_df.to_csv("./sim/obs_df.csv")


# Visualize the initial state
plt.figure(figsize=(10, 6))

# Plot each row with a different color to verify the simulation logic
for row_id in obs_df['Row'].unique():
    row_data = obs_df[obs_df['Row'] == row_id]
    plt.scatter(row_data['Distance'], row_data['Severity'], alpha=0.4, label=f'Row {row_id}')

# Mark the true focus clearly
plt.axvline(focus1[1], color='red', linestyle='--', linewidth=2, label='True Focus 1')
plt.axvline(focus2[1], color='red', linestyle='--', linewidth=2, label='True Focus 2')

plt.title("Simulated Disease Gradient (Raw Data)")
plt.xlabel("Distance from Transect Start")
plt.ylabel("Disease Severity (0 to 1)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('./output/disease_gradient_two_foci.png')

   InterrowDistance  Distance  Severity  Row Plant
0               0.0 -2.000000  0.086257    0     0
1               0.0 -1.783133  0.288976    0     1
2               0.0 -1.566265  0.172174    0     2
3               0.0 -1.349398  0.099978    0     3
4               0.0 -1.132530  0.285350    0     4


## 3. The HiBASIL Model:Priors and Likelihood
We use weakly informed priors to guide the model without forcing a specific answer. Coordinates ($f_x$, $f_y$): Normal distribution centered on the field average. Weight ($w$)
: Beta (2,2) for a mean of 0.5 as start. Likelihood: Zero-Inflated Beta (ZOIB) to handle healthy plants and complete infections simultaneously.

In [3]:
model = hibasil.build_two_foci_model(obs_df, model_type=model_type)

with model:
    print("\n--- Starting MCMC Sampling ---")
    trace = pm.sample(draws=4000, tune=2000, chains=2, target_accept=0.85, random_seed=42, cores=2, nuts_sampler="nutpie")
    
    # Generate Posterior Predictive Check (PPC)
    thin_trace = trace.sel(draw=slice(None, None, 5))  # use every 5th draw
    ppc = pm.sample_posterior_predictive(thin_trace, var_names=['obs'], progressbar=True, random_seed=42)

    # plot posterior predictive check
    y_sim = ppc.posterior_predictive['obs'].values                
    y_obs = obs_df['Severity'].values
    distances = obs_df['Distance'].values
                
    hibasil.plot_posterior_predictive(y_sim, y_obs, distances, model_type, sim=0)                
                        
# calculate ppc metrics
y_pred = ppc.posterior_predictive['obs'].mean(dim=['chain', 'draw']).values
output_dir = './output/'
output_file = output_dir + 'ppc_metrics_all_sims.csv'
pd.DataFrame(columns=['simulation', 'model_type', 'overall_r2', 'overall_rmse', 'overall_mae',
                        'obs_zeros', 'pred_zeros', 'obs_ones', 'pred_ones', 'continuous_r2', 
                        'continuous_rmse']).to_csv(output_file, index=False) # store all ppc metrics
            
hibasil.compute_and_save_metrics(y_obs, y_pred, y_sim, model_type, 0, output_file)


Multi-row model preparation:
- Total observations: 1008
- Number of rows: 4
- Rows: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
- Severity range: [0.0, 1.0]
- Exact zeros: 276 (27.4%)
- Exact ones: 8 (0.8%)
- Continuous: 724 (71.8%)

--- Starting MCMC Sampling ---


Progress,Draws,Divergences,Step Size,Gradients/Draw
,6000,5,0.16,47
,6000,5,0.15,63


Sampling: [obs]


{'simulation': 0,
 'model_type': 'power_law',
 'overall_r2': 0.31701191342945967,
 'overall_rmse': np.float64(0.10381195713729621),
 'overall_mae': 0.051306985088449795,
 'obs_zeros': np.float64(0.27380952380952384),
 'pred_zeros': np.float64(0.2745424107142857),
 'obs_ones': np.float64(0.007936507936507936),
 'pred_ones': np.float64(0.008851066468253968),
 'continuous_r2': 0.6409168872167998,
 'continuous_rmse': np.float64(0.061422771197884606)}

## 4. Results and Spatial Visualization
We evaluate the model's accuracy by mapping the estimated foci against the true coordinates.

In [4]:
# 1. Check Convergence (R-hat should be <= 1.01)
summary = az.summary(trace, var_names=['fx1', 'fy1', 'fz1', 'scale1', 'exponent1', 'fx2', 'fy2', 'fz2', 'scale2', 'exponent2', 'weight'])
print(summary[['mean', 'hdi_3%', 'hdi_97%', 'r_hat']])

# 2. Universal 2D Comparison Plot
# Provides a visual 'Needle in a Haystack' confirmation
obs_data, focus_coords_dict = basil.load_and_process_multi_row_data(obs_df)
basil.plot_multi_row_results(trace, obs_data, focus_coords_dict, y_pred, model_type, sim=0)

print()
print("Congratulations! You have finished Tutorial 2!")

             mean  hdi_3%  hdi_97%  r_hat
fx1[0]      0.392  -2.121    2.506    1.0
fx1[1]      0.131  -1.407    1.584    1.0
fx1[2]      0.170  -1.553    1.748    1.0
fx1[3]      0.181  -1.571    1.816    1.0
fy1[0]     -0.132  -0.958    0.692    1.0
fy1[1]      0.022  -0.568    0.595    1.0
fy1[2]      0.295  -0.387    0.936    1.0
fy1[3]     -0.359  -1.142    0.589    1.0
fz1[0]      0.815   0.615    0.990    1.0
fz1[1]      0.810   0.618    0.990    1.0
fz1[2]      0.791   0.599    0.990    1.0
fz1[3]      0.870   0.676    0.990    1.0
scale1      6.377   2.737   10.373    1.0
exponent1   2.450   1.592    3.318    1.0
fx2[0]      0.559  -2.193    2.591    1.0
fx2[1]      0.145  -1.296    1.510    1.0
fx2[2]      0.304  -1.921    2.193    1.0
fx2[3]      0.400  -2.048    2.335    1.0
fy2[0]     50.482  49.372   51.769    1.0
fy2[1]     49.791  49.109   50.511    1.0
fy2[2]     50.130  49.246   51.148    1.0
fy2[3]     50.319  49.203   51.393    1.0
fz2[0]      0.833   0.623    0.990